# Querying the PDF with RAG

This notebook demonstrates how to query the financial report data stored in ChromaDB using
Retrieval-Augmented Generation (RAG). The query pipeline has three stages:

1. **Retrieve** - Find the most relevant chunks using vector similarity search
2. **Rerank** (optional) - Re-score retrieved chunks with a cross-encoder for better precision
3. **Generate** - Send the retrieved context + question to an LLM for a grounded answer

Prerequisites: Run `02_pdf_ingestion.ipynb` first to populate ChromaDB with embeddings.

## Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.config import load_config
from rag.ingestion.store import ChromaStore
from rag.ingestion.embedder import get_embedder
from rag.retrieval.retriever import retrieve, RetrievedChunk
from rag.retrieval.reranker import rerank
from rag.retrieval.chain import query_rag

In [3]:
config = load_config()
print(f"Embedding provider: {config.active.embedding_provider}")
print(f"Chat provider: {config.active.chat_provider}")
print(f"Retrieval top_k: {config.retrieval.top_k}")
print(f"Rerank enabled: {config.retrieval.rerank}")

Embedding provider: openai
Chat provider: openai
Retrieval top_k: 5
Rerank enabled: True


## Verify ChromaDB has data

Before querying, confirm that the collection has been populated with chunks from the ingestion notebook.

In [4]:
store = ChromaStore(config)
print(f"Collection: {config.chromadb.collection_name}")
print(f"Total chunks stored: {store.count()}")
print(f"\nIngested documents:")
for doc in store.list_documents():
    print(f"  - {doc['file']} ({doc['chunks']} chunks)")

Collection: financial_reports
Total chunks stored: 2763

Ingested documents:
  - 10Q-Q1-2026-as-filed.pdf (343 chunks)
  - 10Q-Q2-2026-as-filed.pdf (540 chunks)
  - GOOG-10-Q-Q1-2026.pdf (649 chunks)
  - form-10-q.pdf (1231 chunks)


## Stage 1: Retrieval (Vector Similarity Search)

The `retrieve()` function:
1. Embeds your question into a vector using the same embedding model used during ingestion
2. Queries ChromaDB for the nearest neighbors (most similar chunks)
3. Filters out low-scoring results below the `score_threshold`
4. Returns the top-k chunks ranked by similarity score

This gives you the raw retrieval results before any LLM is involved.

In [5]:
question = "What was Disney's total revenue for the quarter?"

chunks = retrieve(question, config, top_k=5)
print(f"Retrieved {len(chunks)} relevant chunks\n")

for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} (score: {chunk.score:.4f}) ---")
    print(f"Source: {chunk.metadata.get('source_file')} | "
          f"Page: {chunk.metadata.get('page_number')} | "
          f"Section: {chunk.metadata.get('section_title', '')[:50]}")
    print(f"{chunk.content[:200]}...\n")

Retrieved 5 relevant chunks

--- Chunk 1 (score: 0.7061) ---
Source: form-10-q.pdf | Page: 32 | Section: CURRENT QUARTER RESULTS COMPARED TO PRIOR-YEAR QUA
Revenues for the quarter increased 7%, or $1.5 billion, to $25.2 billion; net income attributable to Disney decreased to $2.2 billion compared to $3.3 billion in the prior-year quarter; and diluted ea...

--- Chunk 2 (score: 0.6575) ---
Source: form-10-q.pdf | Page: 32 | Section: CONSOLIDATED RESULTS
|                                                               | Quarter Ended   | Quarter Ended   | %Change        | Six Months Ended   | Six Months Ended   | %Change        |
|---------------------...

--- Chunk 3 (score: 0.6502) ---
Source: form-10-q.pdf | Page: 4 | Section: THE WALT DISNEY COMPANY CONDENSED CONSOLIDATED STA
|                                                                             | Quarter Ended   | Quarter Ended   | Six Months Ended   | Six Months Ended   |
|-----------------------------------------...

--- Ch

## Stage 2: Reranking (Optional)

Vector similarity is fast but approximate. A cross-encoder reranker reads each (question, chunk)
pair jointly and produces a more accurate relevance score. This is slower but improves precision.

The reranker uses `cross-encoder/ms-marco-MiniLM-L-6-v2` — a small but effective model that
runs locally (no API call needed).

In [6]:
reranked_chunks = rerank(question, chunks, method="cross-encoder", top_k=5)

print(f"Reranked {len(reranked_chunks)} chunks\n")
for i, chunk in enumerate(reranked_chunks, 1):
    print(f"--- Chunk {i} (rerank score: {chunk.score:.4f}) ---")
    print(f"Source: {chunk.metadata.get('source_file')} | "
          f"Page: {chunk.metadata.get('page_number')}")
    print(f"{chunk.content[:200]}...\n")

Reranked 5 chunks

--- Chunk 1 (rerank score: 8.1281) ---
Source: form-10-q.pdf | Page: 32
Revenues for the quarter increased 7%, or $1.5 billion, to $25.2 billion; net income attributable to Disney decreased to $2.2 billion compared to $3.3 billion in the prior-year quarter; and diluted ea...

--- Chunk 2 (rerank score: 4.4170) ---
Source: form-10-q.pdf | Page: 34
Revenues for the current period increased $2.8 billion, to $51.1 billion; net income attributable to Disney decreased $1.2 billion, to $4.6 billion; and EPS decreased to $2.61 from $3.21 in the prior-...

--- Chunk 3 (rerank score: -2.3388) ---
Source: form-10-q.pdf | Page: 5
|                                                                            | Quarter Ended   | Quarter Ended   | Six Months Ended   | Six Months Ended   |
|------------------------------------------...

--- Chunk 4 (rerank score: -3.1349) ---
Source: form-10-q.pdf | Page: 32
|                                                               | Quarter End

## Stage 3: Full RAG Query (Retrieve + Rerank + Generate)

The `query_rag()` function combines all three stages into a single call:
1. Retrieves relevant chunks from ChromaDB
2. Reranks them (if enabled in config)
3. Builds a context string from the top chunks
4. Sends the context + question to the configured chat LLM
5. Returns the answer with source citations and metadata

The LLM is instructed to only answer based on the provided context (no hallucination).

In [7]:
response = query_rag(question, config)

print("Answer:")
print(response.answer)
print(f"\n--- Metadata ---")
print(f"Provider: {response.metadata.provider}")
print(f"Model: {response.metadata.model}")
print(f"Chunks used: {response.metadata.retrieval_count}")
print(f"Latency: {response.metadata.latency_ms}ms")

Answer:
Disney's total revenue for the quarter ended March 28, 2026, was $25.168 billion. This represents a 7% increase compared to the prior-year quarter. [Source 4: form-10-q.pdf, page 32]

--- Metadata ---
Provider: openai
Model: gpt-4o
Chunks used: 5
Latency: 2615ms


In [7]:
print("Sources:")
for i, source in enumerate(response.sources, 1):
    print(f"  [{i}] {source.file}, page {source.page}")
    print(f"      Section: {source.section}")
    print(f"      Excerpt: {source.excerpt[:100]}...\n")

Sources:
  [1] form-10-q.pdf, page 32
      Section: CURRENT QUARTER RESULTS COMPARED TO PRIOR-YEAR QUARTER
      Excerpt: CURRENT QUARTER RESULTS COMPARED TO PRIOR-YEAR QUARTER

Revenues for the quarter increased 7%, or $1...

  [2] form-10-q.pdf, page 32
      Section: CONSOLIDATED RESULTS
      Excerpt: |                                                               | Quarter Ended   | Quarter Ended   ...

  [3] form-10-q.pdf, page 4
      Section: THE WALT DISNEY COMPANY CONDENSED CONSOLIDATED STATEMENTS OF INCOME
      Excerpt: |                                                                             | Quarter Ended   | Qu...



## More Example Queries

Try different types of questions to see how the RAG pipeline handles them.

In [8]:
questions = [
    "What are the main risk factors mentioned in the filing?",
    "What was the operating income for the Entertainment segment?",
    "Did Disney repurchase any shares during the quarter?",
]

for q in questions:
    print(f"Q: {q}")
    resp = query_rag(q, config)
    print(f"A: {resp.answer}\n")
    print(f"   ({resp.metadata.retrieval_count} chunks, "
          f"{resp.metadata.latency_ms}ms)")
    print("=" * 80 + "\n")

Q: What are the main risk factors mentioned in the filing?
A: No relevant information found in the ingested documents.

   (0 chunks, 333ms)

Q: What was the operating income for the Entertainment segment?
A: The operating income for the Entertainment segment was $1,336 million for the quarter ended March 28, 2026, and $2,436 million for the six months ended March 28, 2026 (Source 3: form-10-q.pdf, page 11).

   (5 chunks, 3307ms)

Q: Did Disney repurchase any shares during the quarter?
A: Yes, during the quarter ended March 28, 2026, Disney repurchased 33 million shares of its common stock for $3.5 billion (Source 1: form-10-q.pdf, page 20).

   (3 chunks, 1623ms)



## Filtering by Metadata

ChromaDB supports metadata filtering. This is useful when you have multiple documents
ingested and want to restrict the search to a specific file or page range.

In [ ]:
# Retrieve only from a specific source file
filtered_chunks = retrieve(
    "What is the total debt?",
    config,
    top_k=3,
    where={"source_file": "form-10-q.pdf"},
)

print(f"Retrieved {len(filtered_chunks)} chunks (filtered to form-10-q.pdf)\n")
for i, chunk in enumerate(filtered_chunks, 1):
    print(f"  [{i}] Page {chunk.metadata.get('page_number')} "
          f"(score: {chunk.score:.4f})")
    print(f"      {chunk.content[:100]}...\n")

## Switching Chat Providers

The `query_rag()` function supports switching between configured LLM providers
(OpenAI, Anthropic, Gemini) at query time without changing the config file.

In [ ]:
# Compare answers from different providers
question = "Summarize Disney's financial performance this quarter."

# Using the default provider (from config)
resp_default = query_rag(question, config)
print(f"[{resp_default.metadata.provider} / {resp_default.metadata.model}]")
print(f"{resp_default.answer[:500]}\n")

In [ ]:
# Using a different provider (uncomment the one you want to try)
# resp_anthropic = query_rag(question, config, chat_provider="anthropic")
# print(f"[{resp_anthropic.metadata.provider} / {resp_anthropic.metadata.model}]")
# print(f"{resp_anthropic.answer[:500]}\n")

# resp_gemini = query_rag(question, config, chat_provider="gemini")
# print(f"[{resp_gemini.metadata.provider} / {resp_gemini.metadata.model}]")
# print(f"{resp_gemini.answer[:500]}\n")